In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2 as cv
from PIL import Image

import torch
import torch.nn as nn
from torch.nn import functional as F

import torchvision
from torchvision.models import resnet50, ResNet50_Weights
from torchvision.transforms import v2

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Qurilma: {device}")
print(f"Qurilma nomi: {torch.cuda.get_device_name(0)}")

Qurilma: cuda
Qurilma nomi: Tesla T4


In [3]:
# Pretrained ResNet50 yuklash

weights = ResNet50_Weights.IMAGENET1K_V2
model = resnet50(weights=weights).to(device)

for param in model.parameters():
  param.requires_grad = False

for name, param in model.named_parameters():
    print(
        name,
        "Shape:", param.shape,
        "Device:", param.device,
        "Trainable:", param.requires_grad
    )

Downloading: "https://download.pytorch.org/models/resnet50-11ad3fa6.pth" to /root/.cache/torch/hub/checkpoints/resnet50-11ad3fa6.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 145MB/s]


conv1.weight Shape: torch.Size([64, 3, 7, 7]) Device: cuda:0 Trainable: False
bn1.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
bn1.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv1.weight Shape: torch.Size([64, 64, 1, 1]) Device: cuda:0 Trainable: False
layer1.0.bn1.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.bn1.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv2.weight Shape: torch.Size([64, 64, 3, 3]) Device: cuda:0 Trainable: False
layer1.0.bn2.weight Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.bn2.bias Shape: torch.Size([64]) Device: cuda:0 Trainable: False
layer1.0.conv3.weight Shape: torch.Size([256, 64, 1, 1]) Device: cuda:0 Trainable: False
layer1.0.bn3.weight Shape: torch.Size([256]) Device: cuda:0 Trainable: False
layer1.0.bn3.bias Shape: torch.Size([256]) Device: cuda:0 Trainable: False
layer1.0.downsample.0.weight Shape: torch.Size([256, 64, 1, 1]) Device: c

In [4]:
# Qayta train uchun modeling FC parametrlani trainga ochamiz

model.fc = nn.Linear(in_features = model.fc.in_features, out_features = 10).to(device)

for name, fc in model.fc.named_parameters():
  print(f"Name: {name}, Shape: {fc.shape}, Device: {fc.device}, Trainable: {fc.requires_grad}")

Name: weight, Shape: torch.Size([10, 2048]), Device: cuda:0, Trainable: True
Name: bias, Shape: torch.Size([10]), Device: cuda:0, Trainable: True


In [ ]:
# preprocess = weights.transforms() # Tayyoridan foydalanish uchun

preprocess =  v2.Compose([
    v2.Resize(size=232, interpolation=v2.InterpolationMode.BILINEAR, antialias=True),
    v2.CenterCrop(size=224),
    v2.ToImage(),
    v2.ToDtype(dtype=torch.float32, scale=True),
    v2.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

print(preprocess)

Compose(
      Resize(size=[232], interpolation=InterpolationMode.BILINEAR, antialias=True)
      CenterCrop(size=(224, 224))
      ToImage()
      ToDtype(scale=True)
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225], inplace=False)
)
